# Modelos estatísticos para Copa do Mundo

**Pipeline:** Poisson → Dixon-Coles → ELO → XGBoost → Backtest contra Mundiais passados.

## Aviso honesto sobre o domínio

Copa do Mundo é o cenário **mais hostil** para modelos tipo Poisson:

- Cada seleção joga só 3–7 partidas por edição.
- Seleções fortes raramente se enfrentam fora de torneios.
- Odds históricas detalhadas (Football-Data.co.uk etc.) não cobrem Mundiais bem.
- O mercado é eficiente — *edge* real é pequeno e raro.

**Abordagem adotada:**
1. Treinar Poisson/Dixon-Coles em **todos os jogos de seleções dos últimos N anos** (não só Mundiais), com peso exponencial favorecendo jogos recentes.
2. Calcular ELO incremental jogo a jogo.
3. Holdout: Copa 2018 e/ou 2022.
4. XGBoost usando ELO diff, ranking FIFA, forma recente como features.

## 1. Setup

In [ ]:
# Instale uma vez (descomente se faltar):
# %pip install pandas numpy scipy scikit-learn xgboost matplotlib seaborn requests

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import poisson
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

RNG = np.random.default_rng(42)
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

## 2. Coleta de dados

Usamos o dataset público de [martj42/international_results](https://github.com/martj42/international_results) — todos os jogos oficiais de seleções desde 1872, atualizado periodicamente.

Se você estiver offline, baixe o CSV manualmente e ajuste `DATA_PATH`.

In [ ]:
import requests, os

DATA_URL = 'https://raw.githubusercontent.com/martj42/international_results/master/results.csv'
DATA_PATH = 'results.csv'

if not os.path.exists(DATA_PATH):
    print('Baixando dataset...')
    r = requests.get(DATA_URL, timeout=30)
    r.raise_for_status()
    with open(DATA_PATH, 'wb') as f:
        f.write(r.content)
    print(f'Salvo em {DATA_PATH} ({len(r.content)/1024:.1f} KB)')

df = pd.read_csv(DATA_PATH, parse_dates=['date'])

# IMPORTANTE: o dataset traz jogos FUTUROS ja agendados, sem placar (home_score/away_score = NaN).
# Esses NaN entram na verossimilhanca do Poisson/Dixon-Coles e fazem o otimizador divergir
# (toda previsao colapsa em 0-0). Removemos jogos nao disputados antes de qualquer modelagem.
n_antes = len(df)
df = df.dropna(subset=['home_score', 'away_score']).reset_index(drop=True)
n_removidos = n_antes - len(df)
if n_removidos:
    print(f'Removidos {n_removidos} jogos futuros/sem placar (NaN).')

print(f'{len(df):,} jogos de {df.date.min().date()} a {df.date.max().date()}')
df.head()

## 3. Recorte temporal e features básicas

- Usamos jogos de 2014 em diante para Poisson (4 ciclos de Copa).
- Mundo 2018 e 2022 vão para holdout.

In [ ]:
df = df.sort_values('date').reset_index(drop=True)
df['is_world_cup'] = df['tournament'].str.contains('FIFA World Cup', case=False, na=False)
df['total_goals'] = df['home_score'] + df['away_score']
df['result'] = np.where(df.home_score > df.away_score, 'H',
                np.where(df.home_score < df.away_score, 'A', 'D'))

TRAIN_START = '2014-01-01'
WC_2018 = ('2018-06-14', '2018-07-15')
WC_2022 = ('2022-11-20', '2022-12-18')

train = df[(df.date >= TRAIN_START) & (df.date < WC_2018[0])].copy()
test_2018 = df[(df.date >= WC_2018[0]) & (df.date <= WC_2018[1]) & df.is_world_cup].copy()
train_through_2021 = df[(df.date >= TRAIN_START) & (df.date < WC_2022[0])].copy()
test_2022 = df[(df.date >= WC_2022[0]) & (df.date <= WC_2022[1]) & df.is_world_cup].copy()

print(f'Treino até 2018: {len(train):,} | Holdout WC2018: {len(test_2018)}')
print(f'Treino até 2022: {len(train_through_2021):,} | Holdout WC2022: {len(test_2022)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train.total_goals.value_counts().sort_index().head(12).plot.bar(ax=axes[0])
axes[0].set_title('Distribuição de gols por jogo (treino)')
axes[0].set_xlabel('Gols')
train.result.value_counts(normalize=True).reindex(['H','D','A']).plot.bar(ax=axes[1])
axes[1].set_title('Resultado (mandante na coluna home)')
axes[1].set_ylabel('Frequência')
plt.tight_layout(); plt.show()

print(f'Média gols mandante: {train.home_score.mean():.2f}')
print(f'Média gols visitante: {train.away_score.mean():.2f}')
print(f'% jogos campo neutro: {train.neutral.mean()*100:.1f}%')

## 4. Modelo Poisson independente (baseline)

Para cada time estimamos:
- $\alpha_i$: força ofensiva
- $\beta_i$: força defensiva
- $\gamma$: vantagem de mando (aplicada só quando `neutral=False`)

$$\lambda_{home} = \exp(\alpha_i - \beta_j + \gamma \cdot \mathbb{1}[\text{não neutro}])$$
$$\lambda_{away} = \exp(\alpha_j - \beta_i)$$

Peso temporal exponencial: jogos mais antigos pesam menos ($\xi$ = meia-vida em anos).

In [ ]:
def time_weights(dates, ref_date, half_life_days=365*2):
    """Peso exponencial: jogo de meia-vida atrás pesa 0.5."""
    age_days = (ref_date - pd.to_datetime(dates)).dt.days.values
    return np.exp(-np.log(2) * age_days / half_life_days)

def fit_poisson(df_train, ref_date, half_life_days=730, min_games=5):
    """Maximum likelihood do Poisson com pesos temporais."""
    # filtra times com volume mínimo
    team_games = pd.concat([df_train.home_team, df_train.away_team]).value_counts()
    keep = team_games[team_games >= min_games].index
    d = df_train[df_train.home_team.isin(keep) & df_train.away_team.isin(keep)].copy()

    teams = sorted(set(d.home_team) | set(d.away_team))
    n = len(teams)
    idx = {t: i for i, t in enumerate(teams)}

    home_idx = d.home_team.map(idx).values
    away_idx = d.away_team.map(idx).values
    hg = d.home_score.values
    ag = d.away_score.values
    not_neutral = (~d.neutral).astype(float).values
    w = time_weights(d.date, ref_date, half_life_days)

    # params: [alpha (n), beta (n), home_adv]
    # restrição: sum(alpha) = 0 (identificabilidade) -> aplicada via reparametrização
    def neg_log_lik(params):
        alpha = np.concatenate([params[:n-1], [-params[:n-1].sum()]])
        beta = np.concatenate([params[n-1:2*n-2], [-params[n-1:2*n-2].sum()]])
        home_adv = params[-1]
        lam_h = np.exp(alpha[home_idx] - beta[away_idx] + home_adv * not_neutral)
        lam_a = np.exp(alpha[away_idx] - beta[home_idx])
        ll = w * (hg * np.log(lam_h) - lam_h + ag * np.log(lam_a) - lam_a)
        return -ll.sum()

    x0 = np.zeros(2*n - 1)
    x0[-1] = 0.25  # chute inicial de home advantage
    res = minimize(neg_log_lik, x0, method='L-BFGS-B', options={'maxiter': 500})
    alpha = np.concatenate([res.x[:n-1], [-res.x[:n-1].sum()]])
    beta = np.concatenate([res.x[n-1:2*n-2], [-res.x[n-1:2*n-2].sum()]])
    home_adv = res.x[-1]

    params_df = pd.DataFrame({'team': teams, 'attack': alpha, 'defense': beta})
    return params_df, home_adv, res

params, home_adv, res = fit_poisson(train, ref_date=pd.Timestamp(WC_2018[0]))
print(f'Convergiu: {res.success} | Home advantage: {home_adv:.3f}')
params.sort_values('attack', ascending=False).head(10)

In [ ]:
def match_probs_poisson(home, away, params, home_adv, neutral=True, max_goals=8):
    """Matriz de probabilidades de placar + 1X2."""
    p = params.set_index('team')
    if home not in p.index or away not in p.index:
        return None
    lam_h = np.exp(p.loc[home,'attack'] - p.loc[away,'defense'] + (0 if neutral else home_adv))
    lam_a = np.exp(p.loc[away,'attack'] - p.loc[home,'defense'])
    h_pmf = poisson.pmf(np.arange(max_goals+1), lam_h)
    a_pmf = poisson.pmf(np.arange(max_goals+1), lam_a)
    M = np.outer(h_pmf, a_pmf)
    p_home = np.tril(M, -1).sum()
    p_draw = np.trace(M)
    p_away = np.triu(M, 1).sum()
    # renormaliza pelo truncamento
    s = p_home + p_draw + p_away
    return {'lam_h': lam_h, 'lam_a': lam_a,
            'p_home': p_home/s, 'p_draw': p_draw/s, 'p_away': p_away/s,
            'matrix': M/M.sum()}

# Exemplo: Brasil x Alemanha em campo neutro
ex = match_probs_poisson('Brazil', 'Germany', params, home_adv, neutral=True)
if ex:
    print(f"λ Brasil={ex['lam_h']:.2f} | λ Alemanha={ex['lam_a']:.2f}")
    print(f"P(Brasil)={ex['p_home']:.1%} | P(Empate)={ex['p_draw']:.1%} | P(Alemanha)={ex['p_away']:.1%}")

## 5. Correção Dixon-Coles

Poisson independente subestima placares 0-0, 1-1, 1-0, 0-1. Dixon-Coles aplica fator $\tau(x,y;\lambda,\mu,\rho)$:

$$\tau = \begin{cases}
1 - \lambda\mu\rho & (0,0) \\
1 + \lambda\rho & (0,1) \\
1 + \mu\rho & (1,0) \\
1 - \rho & (1,1) \\
1 & \text{outros}
\end{cases}$$

In [ ]:
def dc_tau(x, y, lam, mu, rho):
    if x == 0 and y == 0: return 1 - lam*mu*rho
    if x == 0 and y == 1: return 1 + lam*rho
    if x == 1 and y == 0: return 1 + mu*rho
    if x == 1 and y == 1: return 1 - rho
    return 1.0

def fit_dixon_coles(df_train, ref_date, half_life_days=730, min_games=5):
    team_games = pd.concat([df_train.home_team, df_train.away_team]).value_counts()
    keep = team_games[team_games >= min_games].index
    d = df_train[df_train.home_team.isin(keep) & df_train.away_team.isin(keep)].copy()

    teams = sorted(set(d.home_team) | set(d.away_team))
    n = len(teams)
    idx = {t: i for i, t in enumerate(teams)}
    home_idx = d.home_team.map(idx).values
    away_idx = d.away_team.map(idx).values
    hg = d.home_score.values.astype(int)
    ag = d.away_score.values.astype(int)
    not_neutral = (~d.neutral).astype(float).values
    w = time_weights(d.date, ref_date, half_life_days)

    def neg_log_lik(params):
        alpha = np.concatenate([params[:n-1], [-params[:n-1].sum()]])
        beta = np.concatenate([params[n-1:2*n-2], [-params[n-1:2*n-2].sum()]])
        home_adv, rho = params[-2], params[-1]
        lam_h = np.exp(alpha[home_idx] - beta[away_idx] + home_adv * not_neutral)
        lam_a = np.exp(alpha[away_idx] - beta[home_idx])
        # log L do Poisson independente
        ll_pois = hg * np.log(lam_h) - lam_h + ag * np.log(lam_a) - lam_a
        # correção Dixon-Coles só nos placares baixos
        tau = np.ones(len(d))
        mask00 = (hg==0)&(ag==0); tau[mask00] = 1 - lam_h[mask00]*lam_a[mask00]*rho
        mask01 = (hg==0)&(ag==1); tau[mask01] = 1 + lam_h[mask01]*rho
        mask10 = (hg==1)&(ag==0); tau[mask10] = 1 + lam_a[mask10]*rho
        mask11 = (hg==1)&(ag==1); tau[mask11] = 1 - rho
        tau = np.clip(tau, 1e-10, None)
        return -(w * (ll_pois + np.log(tau))).sum()

    x0 = np.zeros(2*n)
    x0[-2] = 0.25  # home adv
    x0[-1] = -0.1  # rho
    bounds = [(None,None)]*(2*n-2) + [(0, 1.0), (-0.5, 0.5)]
    res = minimize(neg_log_lik, x0, method='L-BFGS-B', bounds=bounds, options={'maxiter': 500})
    alpha = np.concatenate([res.x[:n-1], [-res.x[:n-1].sum()]])
    beta = np.concatenate([res.x[n-1:2*n-2], [-res.x[n-1:2*n-2].sum()]])
    return pd.DataFrame({'team': teams, 'attack': alpha, 'defense': beta}), res.x[-2], res.x[-1], res

params_dc, ha_dc, rho_dc, res_dc = fit_dixon_coles(train, ref_date=pd.Timestamp(WC_2018[0]))
print(f'Convergiu: {res_dc.success} | home_adv={ha_dc:.3f} | rho={rho_dc:.3f}')
params_dc.sort_values('attack', ascending=False).head(10)

In [ ]:
def match_probs_dc(home, away, params, home_adv, rho, neutral=True, max_goals=8):
    p = params.set_index('team')
    if home not in p.index or away not in p.index:
        return None
    lam_h = np.exp(p.loc[home,'attack'] - p.loc[away,'defense'] + (0 if neutral else home_adv))
    lam_a = np.exp(p.loc[away,'attack'] - p.loc[home,'defense'])
    h_pmf = poisson.pmf(np.arange(max_goals+1), lam_h)
    a_pmf = poisson.pmf(np.arange(max_goals+1), lam_a)
    M = np.outer(h_pmf, a_pmf)
    # aplica tau nos 4 cantos
    M[0,0] *= 1 - lam_h*lam_a*rho
    M[0,1] *= 1 + lam_h*rho
    M[1,0] *= 1 + lam_a*rho
    M[1,1] *= 1 - rho
    M = np.maximum(M, 0); M /= M.sum()
    return {'lam_h': lam_h, 'lam_a': lam_a,
            'p_home': np.tril(M,-1).sum(),
            'p_draw': np.trace(M),
            'p_away': np.triu(M,1).sum(),
            'matrix': M}

## 6. Rating ELO incremental

Versão adaptada do *World Football Elo Ratings*: ajuste por margem de gols + fator de torneio.

In [ ]:
def compute_elo(df_games, k_base=30, home_field=80, init=1500):
    ratings = {}
    history = []
    # peso de torneio (simplificado)
    weight = {'FIFA World Cup': 1.4, 'FIFA World Cup qualification': 1.1,
              'UEFA Euro': 1.3, 'Copa América': 1.3,
              'UEFA Nations League': 1.1, 'Friendly': 0.7}
    for row in df_games.itertuples():
        rh = ratings.get(row.home_team, init)
        ra = ratings.get(row.away_team, init)
        adv = 0 if row.neutral else home_field
        eh = 1 / (1 + 10 ** (-(rh + adv - ra) / 400))
        if row.home_score > row.away_score: sh = 1
        elif row.home_score < row.away_score: sh = 0
        else: sh = 0.5
        diff = abs(row.home_score - row.away_score)
        g = 1 if diff < 2 else (1.5 if diff == 2 else (11 + diff)/8)
        k = k_base * g * weight.get(row.tournament, 1.0)
        history.append({'date': row.date, 'home_team': row.home_team, 'away_team': row.away_team,
                        'elo_home_pre': rh, 'elo_away_pre': ra, 'expected_home': eh})
        ratings[row.home_team] = rh + k * (sh - eh)
        ratings[row.away_team] = ra + k * ((1-sh) - (1-eh))
    return ratings, pd.DataFrame(history)

elo_ratings, elo_hist = compute_elo(train.sort_values('date'))
top_elo = pd.Series(elo_ratings).sort_values(ascending=False).head(15)
top_elo.plot.barh(figsize=(7,5)); plt.gca().invert_yaxis()
plt.title('Top 15 ELO no início da Copa 2018'); plt.tight_layout(); plt.show()

## 7. Features para XGBoost

Para cada jogo, geramos features *causais* (só com info disponível antes da partida).

In [ ]:
def rolling_team_stats(df_all, window=10):
    """Para cada jogo, anexa médias móveis dos últimos N jogos de cada time."""
    df_all = df_all.sort_values('date').reset_index(drop=True).copy()
    # long format por time
    home = df_all[['date','home_team','home_score','away_score']].rename(
        columns={'home_team':'team','home_score':'gf','away_score':'ga'})
    away = df_all[['date','away_team','away_score','home_score']].rename(
        columns={'away_team':'team','away_score':'gf','home_score':'ga'})
    long = pd.concat([home, away]).sort_values('date').reset_index(drop=True)
    long['result_pts'] = np.where(long.gf > long.ga, 3, np.where(long.gf == long.ga, 1, 0))
    # médias móveis com shift pra não vazar o jogo atual
    g = long.groupby('team', group_keys=False)
    long['gf_avg'] = g.gf.apply(lambda s: s.shift().rolling(window, min_periods=3).mean())
    long['ga_avg'] = g.ga.apply(lambda s: s.shift().rolling(window, min_periods=3).mean())
    long['form'] = g.result_pts.apply(lambda s: s.shift().rolling(window, min_periods=3).mean())
    return long

def build_features(df_train_all, df_target):
    long = rolling_team_stats(df_train_all)
    last = long.sort_values('date').groupby('team').tail(1).set_index('team')
    out = df_target.copy()
    for col in ['gf_avg', 'ga_avg', 'form']:
        out[f'h_{col}'] = out.home_team.map(last[col])
        out[f'a_{col}'] = out.away_team.map(last[col])
    # ELO no momento
    final_elo, _ = compute_elo(df_train_all)
    out['h_elo'] = out.home_team.map(final_elo).fillna(1500)
    out['a_elo'] = out.away_team.map(final_elo).fillna(1500)
    out['elo_diff'] = out.h_elo - out.a_elo
    out['gf_diff'] = out.h_gf_avg - out.a_gf_avg
    out['ga_diff'] = out.h_ga_avg - out.a_ga_avg
    out['form_diff'] = out.h_form - out.a_form
    out['neutral_int'] = out.neutral.astype(int)
    return out

feat_2018 = build_features(train, test_2018)
feat_cols = ['elo_diff','gf_diff','ga_diff','form_diff','neutral_int',
             'h_elo','a_elo','h_gf_avg','a_gf_avg','h_ga_avg','a_ga_avg','h_form','a_form']
feat_2018[['home_team','away_team','result'] + feat_cols].head()

## 8. Treino XGBoost

Para ter um conjunto de treino com features causais, construímos features rolantes para todo o período de treino e treinamos um classificador 1X2.

In [ ]:
def build_training_frame(df_period):
    """Gera features causais para cada jogo do período, usando histórico anterior."""
    long = rolling_team_stats(df_period)
    # mapeia (date, team) -> stats antes daquele jogo
    long_keyed = long.set_index(['date','team'])[['gf_avg','ga_avg','form']]
    out = df_period.copy()
    out = out.join(long_keyed, on=['date','home_team']).rename(
        columns={'gf_avg':'h_gf_avg','ga_avg':'h_ga_avg','form':'h_form'})
    out = out.join(long_keyed, on=['date','away_team']).rename(
        columns={'gf_avg':'a_gf_avg','ga_avg':'a_ga_avg','form':'a_form'})
    # ELO incremental por jogo
    _, elo_h = compute_elo(df_period)
    elo_h = elo_h.set_index(['date','home_team','away_team'])[['elo_home_pre','elo_away_pre']]
    out = out.join(elo_h, on=['date','home_team','away_team'])
    out['h_elo'] = out['elo_home_pre']
    out['a_elo'] = out['elo_away_pre']
    out['elo_diff'] = out.h_elo - out.a_elo
    out['gf_diff'] = out.h_gf_avg - out.a_gf_avg
    out['ga_diff'] = out.h_ga_avg - out.a_ga_avg
    out['form_diff'] = out.h_form - out.a_form
    out['neutral_int'] = out.neutral.astype(int)
    return out.dropna(subset=feat_cols)

train_frame = build_training_frame(train)
le = LabelEncoder().fit(['H','D','A'])
X_tr = train_frame[feat_cols].values
y_tr = le.transform(train_frame.result)

clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    objective='multi:softprob', eval_metric='mlogloss',
    subsample=0.85, colsample_bytree=0.85, random_state=42, tree_method='hist')
clf.fit(X_tr, y_tr)
print(f'Treinado em {len(X_tr):,} jogos.')

## 9. Avaliação no holdout (Copa 2018)

Comparamos três previsores nos jogos da fase de grupos (placares + 1X2):

- Poisson simples
- Dixon-Coles
- XGBoost

Métricas: log-loss multiclasse (quanto menor melhor) e Brier.

In [ ]:
rows = []
for r in test_2018.itertuples():
    pp = match_probs_poisson(r.home_team, r.away_team, params, home_adv, neutral=True)
    pd_ = match_probs_dc(r.home_team, r.away_team, params_dc, ha_dc, rho_dc, neutral=True)
    if pp is None or pd_ is None: continue
    rows.append({'date': r.date, 'home': r.home_team, 'away': r.away_team, 'result': r.result,
                 'pois_H': pp['p_home'], 'pois_D': pp['p_draw'], 'pois_A': pp['p_away'],
                 'dc_H': pd_['p_home'], 'dc_D': pd_['p_draw'], 'dc_A': pd_['p_away']})
preds = pd.DataFrame(rows)

# XGBoost no holdout
X_te = feat_2018[feat_cols].values
xgb_probs = clf.predict_proba(X_te)
classes = le.classes_  # ordem do encoder
for i, c in enumerate(classes):
    feat_2018[f'xgb_{c}'] = xgb_probs[:, i]
preds = preds.merge(feat_2018[['home_team','away_team','xgb_H','xgb_D','xgb_A']],
                    left_on=['home','away'], right_on=['home_team','away_team'], how='left')

# métricas
y_true = le.transform(preds.result)
def report(name, P):
    ll = log_loss(y_true, P, labels=[0,1,2])
    acc = accuracy_score(y_true, P.argmax(1))
    print(f'{name:15s} | log-loss={ll:.4f} | acc={acc:.1%}')

# Ordem das colunas: classes do encoder = ['A','D','H']
order = {c: i for i,c in enumerate(classes)}
P_pois = preds[[f'pois_{c}' for c in classes]].values
P_dc   = preds[[f'dc_{c}'   for c in classes]].values
P_xgb  = preds[[f'xgb_{c}'  for c in classes]].dropna().values
preds_valid = preds.dropna(subset=[f'xgb_{c}' for c in classes])
y_valid = le.transform(preds_valid.result)

print('=== Holdout Copa 2018 ===')
report('Poisson',    P_pois)
report('Dixon-Coles', P_dc)
print(f'XGBoost (n={len(P_xgb)}) | log-loss={log_loss(y_valid, P_xgb, labels=[0,1,2]):.4f} | acc={accuracy_score(y_valid, P_xgb.argmax(1)):.1%}')
preds.head(10)

## 10. Coleta de odds históricas (OddsHarvester)

**Realidade do mercado:** pesquisei Football-Data.co.uk, Kaggle, openfootball, e nenhum tem CSV público com odds 1X2 fechadas de Copa do Mundo. O caminho viável é scraping do [oddsportal.com](https://www.oddsportal.com), que arquiva odds históricas de 80+ casas.

[OddsHarvester](https://github.com/jordantete/OddsHarvester) é um wrapper Python sobre Playwright que já tem o slug `world-cup` mapeado em [`sport_league_constants.py`](https://github.com/jordantete/OddsHarvester/blob/main/src/oddsharvester/utils/sport_league_constants.py). **Atenção:** scraping consome banda do oddsportal — use com moderação e respeite os ToS.

### Instalação (terminal, uma vez):
```bash
pip install oddsharvester
playwright install chromium
```

### Comandos para baixar WC 2018 e 2022 (odds 1X2 fechadas):
```bash
oddsharvester historic -s football -l world-cup --season 2018-2018 -m 1x2 --headless --format csv --output wc2018_odds.csv
oddsharvester historic -s football -l world-cup --season 2022-2022 -m 1x2 --headless --format csv --output wc2022_odds.csv
```

Demora ~10–20 min por edição (64 jogos × 80+ casas).

In [ ]:
import subprocess, shutil, os

def scrape_world_cup_odds(season='2022-2022', out_path='wc2022_odds.csv',
                          market='1x2', timeout_sec=1800):
    """Chama OddsHarvester via subprocess. Retorna o caminho do CSV gerado."""
    if shutil.which('oddsharvester') is None:
        raise RuntimeError('OddsHarvester nao esta no PATH. Rode: pip install oddsharvester && playwright install chromium')
    cmd = ['oddsharvester', 'historic', '-s', 'football', '-l', 'world-cup',
           '--season', season, '-m', market, '--headless',
           '--format', 'csv', '--output', out_path]
    print('Rodando:', ' '.join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_sec)
    if res.returncode != 0:
        print('STDERR:', res.stderr[-2000:])
        raise RuntimeError(f'oddsharvester falhou (exit {res.returncode})')
    print(f'OK. CSV em {out_path} ({os.path.getsize(out_path)/1024:.1f} KB)')
    return out_path

# Descomente para executar (demora ~10-20 min cada):
# scrape_world_cup_odds(season='2018-2018', out_path='wc2018_odds.csv')
# scrape_world_cup_odds(season='2022-2022', out_path='wc2022_odds.csv')
print('Helper pronto. Chame scrape_world_cup_odds() quando quiser puxar os dados.')

## 12. Próximos passos

Em ordem de impacto, agora que o pipeline de odds está plugado:

1. **xG por seleção**: integrar dados do FBref (international stats) ou Understat — features muito mais informativas que gols brutos. Times "azarões" que criam muito xG são onde Poisson erra mais.
2. **Calibração**: rodar `CalibratedClassifierCV` (sigmoid ou isotonic) no XGBoost e plotar reliability diagram. Modelo descalibrado destrói qualquer EV — você acha que tem 60% num jogo e na real o "verdadeiro" é 50%, aí o `edge>5%` acende com lixo.
3. **Ensemble**: combinar Dixon-Coles + XGBoost com pesos otimizados em log-loss no holdout (não em acurácia).
4. **Sensibilidade do backtest**: variar `min_edge` (1% a 10%) e `kelly_mult` (1/8 a 1/2) — se ROI só é positivo num ponto exato, é overfit ao holdout.
5. **Estabilidade de nomes**: OddsHarvester pode trazer `Korea Republic` vs `South Korea`, `Iran` vs `Iran, Islamic Republic of`. Crie um dicionário de aliases antes do merge.
6. **Empacotar como tool**: depois de validado em WC 2018 *e* 2022, expor `match_probs_dc` como função de um agente, com cache dos parâmetros treinados.

### Lembrete honesto

Este notebook é ferramenta de **estudo estatístico**. Pinnacle e Betfair têm margens de ~2% em finais de Copa — o *edge* tem que ser maior que isso só pra zerar o vig. ROI positivo num backtest pequeno (1 Copa = ~64 jogos) é dominado por sorte; rode em WC 2018 *e* 2022 *e* Eurocopa pra ter algum sinal. Aposte só com dinheiro que você está disposto a perder.

In [ ]:
def load_odds(path, prefer_book='Pinnacle'):
    """Le CSV do OddsHarvester (ou similar) e normaliza pra home, away, odd_H, odd_D, odd_A."""
    raw = pd.read_csv(path)
    rename = {}
    for c in raw.columns:
        lc = c.lower().strip()
        if lc in ('home_team', 'home', 'team_home'): rename[c] = 'home'
        elif lc in ('away_team', 'away', 'team_away'): rename[c] = 'away'
    raw = raw.rename(columns=rename)

    book_cols = {f'{prefer_book}_home': 'odd_H',
                 f'{prefer_book}_draw': 'odd_D',
                 f'{prefer_book}_away': 'odd_A'}
    if all(c in raw.columns for c in book_cols):
        out = raw[['home','away'] + list(book_cols)].rename(columns=book_cols)
    else:
        candidates = [('home_odds','draw_odds','away_odds'),
                      ('odd_h','odd_d','odd_a'),
                      ('avg_home','avg_draw','avg_away'),
                      ('B365H','B365D','B365A'),
                      ('PSH','PSD','PSA')]
        out = None
        for h,d,a in candidates:
            if all(c in raw.columns for c in (h,d,a)):
                out = raw[['home','away',h,d,a]].rename(
                    columns={h:'odd_H', d:'odd_D', a:'odd_A'})
                break
        if out is None:
            raise ValueError(f'Nao achei colunas de odds em: {list(raw.columns)[:30]}')

    for c in ('odd_H','odd_D','odd_A'):
        out[c] = pd.to_numeric(out[c], errors='coerce')
    return out.dropna(subset=['odd_H','odd_D','odd_A'])

# Carrega se ja tiver scraped
odds_2018 = load_odds('wc2018_odds.csv') if os.path.exists('wc2018_odds.csv') else None
odds_2022 = load_odds('wc2022_odds.csv') if os.path.exists('wc2022_odds.csv') else None
print(f'WC2018: {len(odds_2018) if odds_2018 is not None else 0} jogos | '
      f'WC2022: {len(odds_2022) if odds_2022 is not None else 0} jogos')

### Alternativa manual — entrar odds à mão

Se você só quer validar o pipeline sem rodar scraping, dá pra preencher odds de poucos jogos manualmente (consultando oddsportal.com direto). **Não use estes números sem conferir na fonte — eles são template, não dado.**

In [ ]:
# Template: pegue as odds em oddsportal.com/football/world/world-cup-2022 (aba "Results")
# e preencha as colunas odd_H/odd_D/odd_A. As linhas abaixo sao apenas exemplo de estrutura.
manual_odds_2022 = pd.DataFrame([
    # {'home': 'Argentina', 'away': 'France',  'odd_H': None, 'odd_D': None, 'odd_A': None},  # Final
    # {'home': 'Argentina', 'away': 'Croatia', 'odd_H': None, 'odd_D': None, 'odd_A': None},  # SF
    # {'home': 'France',    'away': 'Morocco', 'odd_H': None, 'odd_D': None, 'odd_A': None},  # SF
])
print(f'manual_odds_2022 com {len(manual_odds_2022)} linhas — preencha antes de usar.')

## 11. Backtest: edge / EV / Kelly fracionário

Com `odds_df` no formato `home, away, odd_H, odd_D, odd_A` e `preds` saindo do Dixon-Coles/XGBoost, computamos:

- **Probabilidade implícita** da odd, removendo a margem (overround) proporcionalmente: $p_{imp} = \frac{1/odd}{\sum 1/odd_i}$.
- **EV** de cada lado: $EV = p_{modelo} \cdot odd - 1$.
- **Stake**: ¼ Kelly sobre EV positivo acima do limite. ¼ Kelly em vez de Kelly cheio pra reduzir variância (Kelly cheio é matematicamente ótimo mas brutal em drawdown).

In [ ]:
def implied_probs(row):
    """Probabilidade implicita removendo overround proporcionalmente."""
    inv = 1 / np.array([row.odd_H, row.odd_D, row.odd_A], dtype=float)
    p = inv / inv.sum()
    return pd.Series({'imp_H': p[0], 'imp_D': p[1], 'imp_A': p[2]})

def kelly_fraction(p, odd):
    b = odd - 1
    f = (b*p - (1-p)) / b
    return max(f, 0)

def backtest(preds_df, odds_df, model_cols=('dc_H','dc_D','dc_A'),
             bankroll=100.0, kelly_mult=0.25, min_edge=0.05):
    """preds_df: home, away, model_cols + result.
       odds_df:  home, away, odd_H, odd_D, odd_A."""
    m = preds_df.merge(odds_df, on=['home','away'], how='inner')
    if m.empty:
        print('Nenhum jogo bateu entre preds e odds. Confira se os nomes dos times sao iguais.')
        return pd.DataFrame(), bankroll
    log = []
    bk = bankroll
    sides = list(zip(['H','D','A'], ['odd_H','odd_D','odd_A'], list(model_cols)))
    for r in m.itertuples():
        for side, odd_col, prob_col in sides:
            odd = getattr(r, odd_col); p = getattr(r, prob_col)
            if not (np.isfinite(odd) and np.isfinite(p)): continue
            ev = p * odd - 1
            if ev > min_edge:
                f = kelly_mult * kelly_fraction(p, odd)
                stake = bk * f
                won = (r.result == side)
                pnl = stake * (odd - 1) if won else -stake
                bk += pnl
                log.append({'home': r.home, 'away': r.away, 'side': side,
                            'odd': odd, 'p_model': p, 'ev': ev, 'stake': stake,
                            'won': won, 'pnl': pnl, 'bankroll': bk})
    return pd.DataFrame(log), bk

# Roda se ja tiver odds carregadas
if odds_2018 is not None and len(preds):
    log, final_bk = backtest(preds, odds_2018, model_cols=('dc_H','dc_D','dc_A'))
    if len(log):
        roi = (final_bk/100 - 1) * 100
        hit = log.won.mean() * 100
        print(f'=== Backtest WC 2018 (Dixon-Coles) ===')
        print(f'Bets: {len(log)} | Hit rate: {hit:.1f}% | Bankroll final: {final_bk:.2f} | ROI: {roi:+.1f}%')
        # curva de bankroll
        log.bankroll.plot(figsize=(9,3), title='Bankroll WC2018 (DC, 1/4 Kelly, edge>5%)')
        plt.axhline(100, ls='--', c='gray'); plt.ylabel('Bankroll'); plt.tight_layout(); plt.show()
        log.head(10)
else:
    print('Sem odds_2018 ainda — rode o scraper na celula 10, ou preencha manual_odds.')

## 12. Foco em amistosos recentes

Amistosos são ruidosos (times escalam reservas, motivação baixa) mas são a fonte principal de info **fora de janelas de qualifiers/copa**. Aqui inspecionamos os amistosos dos últimos 12 meses para entender o sinal que o modelo está absorvendo deles.

In [ ]:
# Filtra amistosos dos ultimos 12 meses
TODAY = df.date.max()  # ultima data no dataset
window_start = TODAY - pd.Timedelta(days=365)
friendlies = df[(df.tournament == 'Friendly') & (df.date >= window_start)].copy()
print(f'{len(friendlies)} amistosos entre {window_start.date()} e {TODAY.date()}')

# Tabela: time x medias ofensivas/defensivas em amistosos
home_stats = friendlies.groupby('home_team').agg(
    games=('date','count'), gf=('home_score','sum'), ga=('away_score','sum'))
away_stats = friendlies.groupby('away_team').agg(
    games=('date','count'), gf=('away_score','sum'), ga=('home_score','sum'))
combined = home_stats.add(away_stats, fill_value=0)
combined['gf_pg'] = combined.gf / combined.games
combined['ga_pg'] = combined.ga / combined.games
top_attack = combined[combined.games >= 4].sort_values('gf_pg', ascending=False).head(15)
print('\n=== Top 15 ataque em amistosos (>= 4 jogos) ===')
print(top_attack[['games','gf_pg','ga_pg']].round(2))

In [ ]:
# Quero o modelo treinado ATE hoje (nao so ate 2018) para previsoes futuras
ref_date_now = TODAY + pd.Timedelta(days=1)
train_full = df[df.date >= '2018-01-01'].copy()  # ultimos ~8 anos
params_full, ha_full, rho_full, _ = fit_dixon_coles(
    train_full, ref_date=ref_date_now, half_life_days=730, min_games=5)
print(f'Modelo retreinado em {len(train_full):,} jogos | home_adv={ha_full:.3f} | rho={rho_full:.3f}')

## 13. Previsão para próximos jogos

### Passo 1 — definir os jogos

Você precisa fornecer a lista de jogos. Tem duas formas:

**a) Manualmente** — preencha o DataFrame `upcoming` abaixo com `home`, `away`, `neutral` (True/False).

**b) De um CSV** — se você tiver `fixtures.csv` com essas colunas, descomente o `pd.read_csv`.

> ⚠️ **Aviso:** o modelo só conhece times que apareceram no treino. Convocações específicas, lesões, suspensões, troca de técnico — *nada disso* entra. Use as probabilidades como uma referência base, não como verdade.

In [ ]:
# a) Manual - edite conforme os jogos que voce quer prever
upcoming = pd.DataFrame([
    {'home': 'Brazil',    'away': 'Argentina', 'neutral': True},
    {'home': 'France',    'away': 'Germany',   'neutral': True},
    {'home': 'Spain',     'away': 'Portugal',  'neutral': True},
    {'home': 'England',   'away': 'Netherlands','neutral': True},
])

# b) De CSV (descomente se tiver fixtures.csv com colunas: home, away, neutral)
# upcoming = pd.read_csv('fixtures.csv')

upcoming

### Passo 2 — probabilidades do modelo

Para cada jogo: probabilidade 1X2, placar mais provável, totals (Over/Under 2.5), Both Teams to Score (BTTS).

In [ ]:
def full_match_report(home, away, params, home_adv, rho, neutral=True, max_goals=8):
    """Retorna 1X2, over/under, btts, placar mais provavel."""
    res = match_probs_dc(home, away, params, home_adv, rho, neutral=neutral, max_goals=max_goals)
    if res is None:
        return None
    M = res['matrix']
    # placar mais provavel
    i, j = np.unravel_index(M.argmax(), M.shape)
    top_score = f'{i}-{j}'
    top_score_prob = M[i, j]
    # over/under 2.5
    total_goals_pmf = np.zeros(2*max_goals + 1)
    for x in range(M.shape[0]):
        for y in range(M.shape[1]):
            total_goals_pmf[x+y] += M[x, y]
    p_over_25 = total_goals_pmf[3:].sum()
    p_under_25 = total_goals_pmf[:3].sum()
    # btts
    p_btts_yes = M[1:, 1:].sum()
    p_btts_no = 1 - p_btts_yes
    return {
        'home': home, 'away': away,
        'lam_h': res['lam_h'], 'lam_a': res['lam_a'],
        'p_H': res['p_home'], 'p_D': res['p_draw'], 'p_A': res['p_away'],
        'top_score': top_score, 'p_top_score': top_score_prob,
        'p_over_25': p_over_25, 'p_under_25': p_under_25,
        'p_btts_yes': p_btts_yes, 'p_btts_no': p_btts_no,
    }

reports = []
missing = []
for r in upcoming.itertuples():
    rep = full_match_report(r.home, r.away, params_full, ha_full, rho_full, neutral=r.neutral)
    if rep is None:
        missing.append((r.home, r.away))
    else:
        reports.append(rep)

if missing:
    print(f'Jogos ignorados (time fora do treino): {missing}')

pred_df = pd.DataFrame(reports)
print('\n=== Probabilidades por jogo ===')
cols_show = ['home','away','p_H','p_D','p_A','top_score','p_top_score','p_over_25','p_btts_yes']
print(pred_df[cols_show].round(3).to_string(index=False))

### Passo 3 — duas leituras diferentes

**A. "Resultado mais provável" (info, NÃO recomendação de aposta)** — qual a aposta com maior chance de bater. Apostar nisso sempre perde no longo prazo porque a casa já cobra a margem.

**B. "Melhor aposta" = maior EV** — só faz sentido quando você compara *sua* probabilidade com a *odd* da casa. Sem odd, não há "melhor aposta".

In [ ]:
# A) Resultado mais provavel por jogo (info)
def most_likely_pick(row):
    options = {
        f'{row.home} vence (1)': row.p_H,
        'Empate (X)': row.p_D,
        f'{row.away} vence (2)': row.p_A,
        f'Over 2.5 gols': row.p_over_25,
        f'Under 2.5 gols': row.p_under_25,
        f'Ambos marcam (Sim)': row.p_btts_yes,
        f'Ambos marcam (Nao)': row.p_btts_no,
    }
    pick, prob = max(options.items(), key=lambda kv: kv[1])
    return pick, prob

print('=== Resultado mais provavel por jogo (info, NAO recomendacao) ===')
for r in pred_df.itertuples():
    pick, prob = most_likely_pick(r)
    print(f'{r.home:>15} x {r.away:<15} | {pick:35} ({prob*100:.1f}%) | placar mais prov: {r.top_score}')

In [ ]:
# B) Ranking por EV - exige odds atuais do mercado
# Formato esperado de upcoming_odds: home, away, odd_H, odd_D, odd_A
upcoming_odds = pd.DataFrame([
    # Preencha com as odds atuais (oddsportal, bet365, pinnacle...)
    # {'home': 'Brazil',  'away': 'Argentina', 'odd_H': 2.30, 'odd_D': 3.20, 'odd_A': 3.10},
    # {'home': 'France',  'away': 'Germany',   'odd_H': 2.40, 'odd_D': 3.10, 'odd_A': 3.00},
])

if not upcoming_odds.empty:
    merged = pred_df.merge(upcoming_odds, on=['home','away'], how='inner')
    bets = []
    for r in merged.itertuples():
        for side, odd, p in [('Home', r.odd_H, r.p_H),
                              ('Draw', r.odd_D, r.p_D),
                              ('Away', r.odd_A, r.p_A)]:
            implied = 1/odd
            ev = p * odd - 1
            bets.append({'jogo': f'{r.home} x {r.away}', 'aposta': side,
                         'odd': odd, 'p_modelo': p, 'p_implicita': implied,
                         'edge': p - implied, 'EV': ev})
    bets_df = pd.DataFrame(bets).sort_values('EV', ascending=False)
    print('=== Ranking por EV (positivo = +EV; negativo = casa em vantagem) ===')
    print(bets_df.round(3).to_string(index=False))
    print('\nSugestao: so apostar em EV > 0.05 (5%) E com a casa tendo margem baixa (Pinnacle, Betfair).')
else:
    print('Preencha upcoming_odds com as odds atuais para ver o ranking de EV.')
    print('Sem odds, "melhor aposta" eh impossivel de calcular - apenas "mais provavel" (que NAO eh a mesma coisa).')

## 11. Próximos passos

Em ordem de impacto:

1. **Odds históricas**: localizar CSV de odds para WC 2018/2022 (oddsportal scraping, ou datasets do Kaggle como `world-cup-2022-fifa`). Sem isso, o conceito de *edge* fica abstrato.
2. **xG por seleção**: integrar dados do FBref (international stats) — features bem mais informativas que gols brutos.
3. **Calibração**: rodar `CalibratedClassifierCV` no XGBoost e checar reliability diagram. Modelo descalibrado destrói qualquer EV.
4. **Ensemble**: combinar Dixon-Coles + XGBoost com pesos otimizados no holdout.
5. **Empacotar como tool**: depois de validado, expor `match_probs_dc` como função de um agente, com cache dos parâmetros.

### Lembrete honesto
Este notebook é ferramenta de **estudo estatístico**. Não existe previsão garantida; o mercado é eficiente e *edge* real costuma ser pequeno e raro. Aposte só com dinheiro que você está disposto a perder.